In [5]:
import torch
import torchvision
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from torchvision import transforms

writer = SummaryWriter('../logs')

In [6]:
train_data = torchvision.datasets.CIFAR10('../dataset', train=True, transform=transforms.ToTensor())
test_data = torchvision.datasets.CIFAR10('../dataset', train=False, transform=transforms.ToTensor())

train_size = len(train_data)
test_size = len(test_data)

train_dataloader = DataLoader(dataset=train_data, batch_size=64)
test_dataloader = DataLoader(dataset=test_data, batch_size=64)

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

In [7]:
class MyModule(nn.Module):
    def __init__(self):
        super(MyModule, self).__init__()
        self.model = nn.Sequential(
            nn.Conv2d(3, 32, 5, padding=2),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 32, 5, padding=2),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 5, padding=2),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, 64),
            nn.Linear(64, 10)
        )

    def forward(self, x):
        return self.model(x)

In [8]:
my_module = MyModule()
my_module.to(device)

loss_fn = nn.CrossEntropyLoss()
loss_fn.to(device)

optim = torch.optim.SGD(my_module.parameters(), lr=1e-2)

train_step = 0
test_step = 0

train_epoch = 20

for epoch in range(train_epoch):
    my_module.train()
    for data in train_dataloader:
        imgs, targets = data
        imgs = imgs.to(device)
        targets = targets.to(device)
        outputs = my_module(imgs)
        loss = loss_fn(outputs, targets)

        optim.zero_grad()
        loss.backward()
        optim.step()

        train_step += 1
        if train_step % 100:
            writer.add_scalar('train_loss', loss.item(), train_step)

    my_module.eval()
    test_loss = 0
    test_accurate = 0
    with torch.no_grad():
        for data in test_dataloader:
            imgs, targets = data
            imgs = imgs.to(device)
            targets = targets.to(device)
            outputs = my_module(imgs)
            loss = loss_fn(outputs, targets)

            test_loss += loss.item()
            test_accurate += (torch.argmax(outputs, dim=1) == targets).sum()

    print(f'epoch: %d, test_loss: %.4f, test_accurate: %.4f' % (epoch, test_loss, test_accurate/test_size))
    writer.add_scalar('test_loss', test_loss, test_step)
    writer.add_scalar('test_accurate', test_accurate/test_size, test_step)
    test_step += 1

epoch: 0, test_loss: 356.8148, test_accurate: 0.1385
epoch: 1, test_loss: 315.5877, test_accurate: 0.2838
epoch: 2, test_loss: 283.0992, test_accurate: 0.3696
epoch: 3, test_loss: 258.5403, test_accurate: 0.4060
epoch: 4, test_loss: 249.5725, test_accurate: 0.4194
epoch: 5, test_loss: 247.7411, test_accurate: 0.4275
epoch: 6, test_loss: 236.4616, test_accurate: 0.4533
epoch: 7, test_loss: 227.2828, test_accurate: 0.4741
epoch: 8, test_loss: 218.7491, test_accurate: 0.4938
epoch: 9, test_loss: 206.6126, test_accurate: 0.5253
epoch: 10, test_loss: 196.4430, test_accurate: 0.5511
epoch: 11, test_loss: 189.6471, test_accurate: 0.5719
epoch: 12, test_loss: 184.0546, test_accurate: 0.5857
epoch: 13, test_loss: 179.8005, test_accurate: 0.5953
epoch: 14, test_loss: 175.8920, test_accurate: 0.6054
epoch: 15, test_loss: 173.8191, test_accurate: 0.6084
epoch: 16, test_loss: 172.3251, test_accurate: 0.6112
epoch: 17, test_loss: 170.0057, test_accurate: 0.6185
epoch: 18, test_loss: 167.7821, test_a

In [9]:
writer.close()